# Insurance Claims Analytics & Risk Assessment
## Module 5 — Customer Risk Profiling

**Business Objective**
Construct a risk profile for every policyholder to support underwriting
decisions, renewal pricing strategy, and portfolio risk management.

**Risk Scoring Model**
The composite risk score is a weighted combination of three normalised components:

| Component | Weight | Rationale |
|---|---|---|
| Claim Amount | 40% | Primary indicator of direct financial exposure |
| Age | 30% | Actuarial risk proxy — older policyholders carry higher claim probability |
| Utilisation | 30% | Measures how much of the coverage has been consumed |

K-Means clustering independently validates the scoring model by identifying
naturally occurring risk groupings without predefined thresholds.


In [ ]:
import sys
sys.path.insert(0, '..')
from src.risk_profiling import (
    load_cleaned_data, compute_risk_score, assign_risk_segments,
    run_kmeans_clustering, plot_risk_segmentation, plot_kmeans_clusters,
    save_risk_profiles
)
from IPython.display import Image
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

df = load_cleaned_data()


## 5.1 Risk Score Computation

In [ ]:
df = compute_risk_score(df)
print(f"Risk score range : {df['RiskScore'].min():.4f} – {df['RiskScore'].max():.4f}")
print(f"Mean risk score  : {df['RiskScore'].mean():.4f}")
df[['CustomerID', 'Age', 'ClaimAmount', 'Utilization', 'RiskScore']].sort_values('RiskScore', ascending=False).head(10)


## 5.2 Risk Segment Assignment

In [ ]:
df = assign_risk_segments(df)
print("Risk Segment Distribution:")
print(df['RiskSegment'].value_counts().to_string())
print(f"\nAverage risk score by segment:")
print(df.groupby('RiskSegment')['RiskScore'].mean().round(4).to_string())


## 5.3 K-Means Clustering Validation

In [ ]:
df = run_kmeans_clustering(df)
print("K-Means Segment Distribution:")
print(df['KMeansSegment'].value_counts().to_string())
print(f"\nSegment Agreement Rate:")
agreement = (df['RiskSegment'].astype(str) == df['KMeansSegment'].astype(str)).mean()
print(f"{agreement:.1%} of records assigned to same tier by both methods")


## 5.4 Risk Segmentation Visualisation

In [ ]:
plot_risk_segmentation(df)
Image('../visuals/risk_segmentation.png')


In [ ]:
plot_kmeans_clusters(df)
Image('../visuals/kmeans_clusters.png')


## 5.5 Risk Segment Financial Profile

In [ ]:
profile = df.groupby('RiskSegment').agg(
    count=('RiskScore', 'count'),
    avg_risk_score=('RiskScore', 'mean'),
    avg_age=('Age', 'mean'),
    avg_premium=('PremiumAmount', 'mean'),
    avg_claim=('ClaimAmount', 'mean'),
    avg_utilization=('Utilization', 'mean'),
    total_exposure=('ClaimAmount', 'sum')
).round(2)
print("Risk Segment Financial Profile:")
print(profile.to_string())


## 5.6 Top 100 High-Risk Customers

In [ ]:
save_risk_profiles(df)
top100 = df[df['RiskSegment'] == 'High Risk'].sort_values('RiskScore', ascending=False).head(10)
cols = ['CustomerID', 'Age', 'PolicyType', 'PremiumAmount', 'ClaimAmount', 'Utilization', 'RiskScore', 'ClaimStatus']
print("Top 10 Highest Risk Customers:")
top100[cols]


---
**Next Module →** `06_Claim_Prediction.ipynb`
